
# Final Merged Dataset Inspection

This notebook inspects the final merged CSV:

- structural integrity
- missingness
- merge/context quality
- thread consistency
- text quality
- stock coverage
- most talked stocks

It is designed as a reproducible audit notebook for the final merged dataset before LLM steps.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

FINAL_DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'wsb_merged_comments_with_submission.csv'


## 1. Load dataset

In [3]:

df = pd.read_csv(FINAL_DATASET_PATH, low_memory=False)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(5))


Shape: (497867, 11)
Columns: ['id', 'submission_id', 'thread_position', 'created_utc', 'author', 'score', 'message_text', 'submission_text', 'matched_tickers', 'match_count', 'needs_context_filter']


,id,submission_id,thread_position,created_utc,author,score,message_text,submission_text,matched_tickers,match_count,needs_context_filter
0,mcsiezy,10up2n6,1,1.739565e+09,BraveSquirrel,1,"Nice buy bro, hope you held on to it and you'd be way up. I'm just coming back from a remindme comment to make fun of people telling you you were an idiot for buying tsla lol",Finally caved in &amp; bought TSLA (1st time). I know RH sucks but I keep all my risky investments on it (I use Fidelity for other stuff),TSLA,1,0
1,n18xdz4,11knpf0,1,1.751601e+09,newimagez,1,Here on 07/03/2025. NVDA hits ATH.,Robinhood Stock Lending Experience\n\nTLDR: How much can you make? About 0.1%. Not 1% -- 0.1% annually. Almost nothing.\n\nI'm sharing my experience with Robinhood Stock Lending because it was ver...,NVDA,1,0
2,nhlf82f,11ugvmt,1,1.759518e+09,Commercial-Employ263,1,"Cramer is an old school fundamentalist and his career is a tv personality, a book writer and a club membership president who charges people to hear hackneyed obvious ""advice"" like ""do your homewor...",Jim Cramer is a moron who agrees?\n\nI was listening to him just a few days ago and he was telling people to steer clear of Bitcoin when it was around $19-$20k it since has outperformed the market...,NVDA,1,0
3,mrxpofk,11zvsca,1,1.747067e+09,Smoogs2,1,Looks like NVDA is projected to beat that JPM 2025 revenue number. Want to revisit our discussion? It's been a crazy two years!,NVDA is up 74% since ChatGPT release (152% since Oct bottom). Overvalued by ANY metric. What will trigger a repricing?,NVDA,1,0
4,mtawcup,14mcdzo,1,1.747752e+09,tke248,1,"I don’t know why the stock price is what it is currently, best advice I can give is take all available data to inform your decision if you still believe keep acquiring more shares until the market...",AbCellera (ABCL): The AI-Powered Drug Discovery Gem You've Been Snoozing On - An Easy 2x Play - DD\n\nGet set for a deep-sea venture into a biotech pearl that's waiting to be discovered in our ver...,PLTR,1,0


## 2. Schema and dtypes

In [5]:

schema = pd.DataFrame({
    "non_null": df.notna().sum(),
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(dropna=False)
})
display(schema)


,non_null,dtype,n_unique
id,497867,object,497867
submission_id,497867,object,14084
thread_position,497867,int64,3264
created_utc,497867,float64,486236
author,497867,object,62051
score,497867,int64,936
message_text,497867,object,484318
submission_text,497866,object,14009
matched_tickers,497867,object,414
match_count,497867,int64,9


## 3. Missingness audit

In [6]:

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(4)
}).sort_values("missing_pct", ascending=False)

display(missing)


,missing_count,missing_pct
submission_text,1,0.0002
submission_id,0,0.0000
id,0,0.0000
thread_position,0,0.0000
created_utc,0,0.0000
score,0,0.0000
author,0,0.0000
message_text,0,0.0000
matched_tickers,0,0.0000
match_count,0,0.0000


## 4. Key integrity checks

In [5]:

checks = {
    "duplicate_comment_ids": int(df.duplicated(subset=["id"]).sum()),
    "missing_submission_id": int(df["submission_id"].isna().sum()),
    "missing_submission_text": int(df["submission_text"].isna().sum()),
    "missing_message_text": int(df["message_text"].isna().sum()),
    "missing_thread_position": int(df["thread_position"].isna().sum()),
    "comment_ids_unique": bool(df["id"].is_unique),
}
checks


{'duplicate_comment_ids': 0,
 'missing_submission_id': 0,
 'missing_submission_text': 1,
 'missing_message_text': 0,
 'missing_thread_position': 0,
 'comment_ids_unique': True}

## 5. Timestamp sanity checks

In [6]:

df["created_dt"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")
print("Min datetime:", df["created_dt"].min())
print("Max datetime:", df["created_dt"].max())

display(df[["created_utc", "created_dt"]].head(10))


Min datetime: 2025-01-01 00:02:17
Max datetime: 2025-12-31 23:59:57


,created_utc,created_dt
0,1.739565e+09,2025-02-14 20:35:46
1,1.751601e+09,2025-07-04 03:43:17
2,1.759518e+09,2025-10-03 18:58:27
3,1.747067e+09,2025-05-12 16:22:26
4,1.747752e+09,2025-05-20 14:37:14
5,1.739658e+09,2025-02-15 22:19:35
6,1.752267e+09,2025-07-11 20:44:31
7,1.752144e+09,2025-07-10 10:33:49
8,1.761703e+09,2025-10-29 01:53:52
9,1.761645e+09,2025-10-28 09:53:15


## 6. Thread ordering and thread-position consistency

In [7]:

ordered_df = df.sort_values(["submission_id", "created_utc", "thread_position"]).copy()

def count_bad_time_order(data):
    bad = 0
    for _, g in data.groupby("submission_id"):
        if not g["created_utc"].is_monotonic_increasing:
            bad += 1
    return bad

def count_bad_thread_position(data):
    bad = 0
    for _, g in data.groupby("submission_id"):
        expected = list(range(1, len(g) + 1))
        observed = g["thread_position"].tolist()
        if observed != expected:
            bad += 1
    return bad

print("Threads with bad time order:", count_bad_time_order(df))
print("Threads with bad time order after sort:", count_bad_time_order(ordered_df))
print("Threads with bad thread_position:", count_bad_thread_position(ordered_df))


Threads with bad time order: 0
Threads with bad time order after sort: 0
Threads with bad thread_position: 0


## 7. Submission-context quality

In [8]:

sub_context = pd.DataFrame({
    "submission_text_empty_string": [(df["submission_text"].fillna("").astype(str).str.strip() == "").sum()],
    "submission_text_deleted_tag": [df["submission_text"].fillna("").astype(str).str.strip().str.fullmatch(r"\[deleted\]", case=False).sum()],
    "submission_text_removed_tag": [df["submission_text"].fillna("").astype(str).str.strip().str.fullmatch(r"\[removed\]", case=False).sum()],
    "submission_text_very_short_le_10": [(df["submission_text"].fillna("").astype(str).str.strip().str.len() <= 10).sum()],
})
display(sub_context.T.rename(columns={0: "count"}))


,count
submission_text_empty_string,1
submission_text_deleted_tag,0
submission_text_removed_tag,0
submission_text_very_short_le_10,659


## 8. Message-text quality

In [9]:

msg = df["message_text"].fillna("").astype(str).str.strip()

message_quality = pd.DataFrame({
    "empty_string": [(msg == "").sum()],
    "deleted_tag": [msg.str.fullmatch(r"\[deleted\]", case=False).sum()],
    "removed_tag": [msg.str.fullmatch(r"\[removed\]", case=False).sum()],
    "very_short_le_5": [(msg.str.len() <= 5).sum()],
    "contains_img_marker": [msg.str.contains(r"\[img\]", case=False, regex=True).sum()],
    "contains_emote_marker": [msg.str.contains(r"emote", case=False, regex=True).sum()],
}).T.rename(columns={0: "count"})

display(message_quality)


,count
empty_string,0
deleted_tag,0
removed_tag,0
very_short_le_5,2238
contains_img_marker,40088
contains_emote_marker,40590


## 9. Duplicate-content checks

In [10]:

dup_checks = {
    "duplicate_message_text": int(df.duplicated(subset=["message_text"]).sum()),
    "duplicate_submission_message_pair": int(df.duplicated(subset=["submission_id", "message_text"]).sum()),
    "duplicate_submission_time_message": int(df.duplicated(subset=["submission_id", "created_utc", "message_text"]).sum()),
}
dup_checks


{'duplicate_message_text': 13549,
 'duplicate_submission_message_pair': 2140,
 'duplicate_submission_time_message': 0}

## 10. Thread size distribution

In [11]:

thread_sizes = df.groupby("submission_id").size().sort_values(ascending=False)
display(thread_sizes.describe())

print("Top 10 largest threads:")
display(thread_sizes.head(10).to_frame("n_comments"))


count    14084.000000
mean        35.349830
std        173.592872
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max       3264.000000
dtype: float64

Top 10 largest threads:


,n_comments
submission_id,
1ib5x15,3264
1idkgpo,2933
1id4ese,2763
1j8n736,2757
1k5fvyp,2742
1ihsa6p,2602
1k7h52x,2579
1j7v45w,2515
1l3v9pt,2485


## 11. Inspect one large thread manually

In [11]:

largest_submission_id = thread_sizes.index[0]
sample_thread = (
    df[df["submission_id"].astype(str) == str(largest_submission_id)]
    .sort_values(["created_utc", "thread_position"])
    .copy()
)

print("Largest submission_id:", largest_submission_id)
display(sample_thread[[
    "id", "submission_id", "thread_position", "created_utc", "author",
    "message_text", "submission_text", "matched_tickers", "match_count",
    "needs_context_filter"
]].head(30))


NameError: name 'thread_sizes' is not defined

## 12. Coverage of submission context per thread

In [13]:

thread_context_coverage = (
    df.assign(has_submission_text=df["submission_text"].notna() & (df["submission_text"].astype(str).str.strip() != ""))
      .groupby("submission_id")["has_submission_text"]
      .agg(["mean", "sum", "count"])
      .rename(columns={"mean": "context_coverage_pct", "sum": "rows_with_context", "count": "thread_size"})
)

display(thread_context_coverage.sort_values("context_coverage_pct").head(20))


,context_coverage_pct,rows_with_context,thread_size
submission_id,,,
v0o5nl,0.0,0,1
w40mbn,1.0,1,1
uu4mtw,1.0,2,2
uh74n2,1.0,3,3
tsxp9m,1.0,1,1
tndnwh,1.0,1,1
stm4ru,1.0,1,1
r38nn2,1.0,4,4
pt48ds,1.0,1,1


## 13. Most talked stocks

In [14]:

stocks_raw = df["matched_tickers"].fillna("").astype(str).str.strip()

stocks_exploded = (
    stocks_raw.replace("", np.nan)
    .dropna()
    .str.split(r"[;,|]")
    .explode()
    .astype(str)
    .str.strip()
)

stocks_exploded = stocks_exploded[stocks_exploded != ""]
stock_counts = stocks_exploded.value_counts().rename_axis("ticker").reset_index(name="n_rows")

print("Most talked stocks by row count:")
display(stock_counts.head(20))


Most talked stocks by row count:


,ticker,n_rows
0,TSLA,187435
1,NVDA,162553
2,AMD,44958
3,PLTR,41254
4,INTC,30829
5,MSFT,21719
6,GOOGL,19839
7,AAPL,12150
8,META,7186
9,AMZN,6791


## 14. Most talked stocks by unique threads

In [15]:

ticker_thread_pairs = df[["submission_id", "matched_tickers"]].copy()
ticker_thread_pairs["matched_tickers"] = ticker_thread_pairs["matched_tickers"].fillna("").astype(str).str.strip()
ticker_thread_pairs = ticker_thread_pairs[ticker_thread_pairs["matched_tickers"] != ""].copy()
ticker_thread_pairs["ticker"] = ticker_thread_pairs["matched_tickers"].str.split(r"[;,|]")
ticker_thread_pairs = ticker_thread_pairs.explode("ticker")
ticker_thread_pairs["ticker"] = ticker_thread_pairs["ticker"].astype(str).str.strip()
ticker_thread_pairs = ticker_thread_pairs[ticker_thread_pairs["ticker"] != ""]

stock_thread_counts = (
    ticker_thread_pairs.drop_duplicates(subset=["submission_id", "ticker"])
    .groupby("ticker")
    .size()
    .sort_values(ascending=False)
    .rename("n_threads")
    .reset_index()
)

print("Most talked stocks by unique threads:")
display(stock_thread_counts.head(20))


Most talked stocks by unique threads:


,ticker,n_threads
0,TSLA,6664
1,NVDA,6623
2,PLTR,3329
3,AMD,2910
4,INTC,2795
5,GOOGL,2670
6,MSFT,2512
7,AAPL,1950
8,AMZN,1627
9,META,1539


## 15. Potential flaws worth flagging

In [16]:

flags = {
    "rows_without_submission_context": int(df["submission_text"].isna().sum()),
    "rows_without_message_text": int(df["message_text"].isna().sum()),
    "rows_with_very_short_message_le_5": int((df["message_text"].fillna("").astype(str).str.strip().str.len() <= 5).sum()),
    "rows_with_img_marker": int(df["message_text"].fillna("").astype(str).str.contains(r"\[img\]", case=False, regex=True).sum()),
    "rows_with_emote_marker": int(df["message_text"].fillna("").astype(str).str.contains(r"emote", case=False, regex=True).sum()),
    "duplicate_comment_ids": int(df.duplicated(subset=["id"]).sum()),
}
flags


{'rows_without_submission_context': 1,
 'rows_without_message_text': 0,
 'rows_with_very_short_message_le_5': 2238,
 'rows_with_img_marker': 40088,
 'rows_with_emote_marker': 40590,
 'duplicate_comment_ids': 0}


## 16. Interpretation checklist

After running this notebook, you should be able to answer clearly:

1. Is the final merged dataset structurally complete?
2. Did the merge leave comments without submission context?
3. Is thread ordering correct?
4. Is `thread_position` consistent and usable for previous-message windows?
5. How much residual text noise remains (`[img]`, emotes, very short comments)?
6. Which stocks dominate the corpus?
7. Are there extreme threads that make raw full-thread prompting impractical?
8. Is the dataset ready for:
   - thread-level context summarization
   - row-level relevance labeling
   - later topic modeling
